# 3. Explainability, Análisis de Errores y Conclusiones

Evaluación del modelo final (V4) en el test set, visualización de Grad-CAM, análisis de errores y reflexión sobre limitaciones y posibles mejoras.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import load_config, get_device
from src.utils.visualization import set_style, plot_predictions_vs_actual
from src.data import create_dataloaders, prepare_metadata, create_splits, extract_text_features
from src.models import WatchPriceCNN
from src.evaluation import predict, compute_metrics, print_metrics
from src.explainability import generate_gradcam, plot_gradcam_grid, visualize_first_layer_filters

set_style()
config = load_config('../configs/base.yaml')
config['data']['metadata_path'] = '../data/metadata.csv'
config['data']['root_dir'] = '../data/raw'
device = get_device(config)

## 3.1 Cargar modelo final

In [ ]:
_, _, test_loader, brand2idx = create_dataloaders(config)
config['model']['num_brands'] = len(brand2idx)

checkpoint = torch.load('../outputs/checkpoints/best_model.pt', map_location=device, weights_only=False)
model = WatchPriceCNN(config['model'])
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
print(f'Modelo cargado: epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.4f}')

## 3.2 Evaluación en test set

In [ ]:
y_true, y_pred = predict(model, test_loader, device)
metrics = compute_metrics(y_true, y_pred, log_transformed=True)
print_metrics(metrics)

In [ ]:
y_true_dollar = np.expm1(y_true)
y_pred_dollar = np.expm1(y_pred)

fig = plot_predictions_vs_actual(y_true_dollar, y_pred_dollar)
plt.savefig('../outputs/figures/pred_vs_actual_final.png', dpi=150)

## 3.3 Grad-CAM: ¿Qué mira el modelo?

Grad-CAM visualiza las regiones de la imagen que más influyen en la predicción del precio. Usamos la última capa convolucional como target.

In [ ]:
images, brand_idxs, text_feats, targets = next(iter(test_loader))
n = min(25, len(images))

cams = generate_gradcam(model, images[:n], device=device)

with torch.no_grad():
    preds = model(images[:n].to(device), brand_idxs[:n].to(device), text_feats[:n].to(device)).cpu().numpy()

plot_gradcam_grid(
    images[:n], cams,
    predictions=np.expm1(preds),
    actuals=np.expm1(targets[:n].numpy()),
)
plt.savefig('../outputs/figures/gradcam_final.png', dpi=150)

**Interpretación del Grad-CAM**:
- El modelo atiende a la **esfera completa** del reloj, no a puntos aislados
- Presta especial atención al **bisel**, las **manecillas** y las **subesferas** (chronograph)
- En relojes con brazalete metálico, también activa en los **eslabones**
- En smartwatches, activa en la **pantalla digital**

Esto confirma que la CNN ha aprendido features visualmente relevantes para estimar el precio: complejidad de la esfera, tipo de correa/brazalete, y acabado general.

## 3.4 Filtros de primera capa

In [ ]:
fig = visualize_first_layer_filters(model)
plt.savefig('../outputs/figures/first_layer_filters_final.png', dpi=150)

Los 32 filtros de la primera capa muestran detección de bordes, contrastes de color y patrones de textura. Estos son los bloques fundamentales que la red utiliza para construir representaciones más complejas en capas posteriores.

## 3.5 Análisis de errores

In [ ]:
# Reconstruir test_df con predicciones
df = prepare_metadata(config)
_, _, test_df = create_splits(df, config)
test_df = test_df.reset_index(drop=True)
test_df['pred_dollar'] = y_pred_dollar
test_df['error'] = test_df['pred_dollar'] - test_df['price_clean']
test_df['abs_error'] = np.abs(test_df['error'])
test_df['pct_error'] = test_df['abs_error'] / test_df['price_clean'] * 100

print('=== Top 10 peores predicciones ===')
worst = test_df.nlargest(10, 'abs_error')[['brand', 'name', 'price_clean', 'pred_dollar', 'error']]
worst.columns = ['Marca', 'Nombre', 'Precio Real', 'Predicción', 'Error']
worst['Precio Real'] = worst['Precio Real'].apply(lambda x: f'${x:.0f}')
worst['Predicción'] = worst['Predicción'].apply(lambda x: f'${x:.0f}')
worst['Error'] = worst['Error'].apply(lambda x: f'${x:+.0f}')
worst

In [ ]:
# Error por rango de precio
test_df['price_range'] = pd.cut(test_df['price_clean'], 
    bins=[0, 100, 250, 500, 1000, 5000], 
    labels=['<$100', '$100-250', '$250-500', '$500-1K', '$1K+'])

error_by_range = test_df.groupby('price_range').agg(
    count=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    mape=('pct_error', 'mean'),
).round(1)
error_by_range.columns = ['N muestras', 'MAE ($)', 'MAPE (%)']
print('=== Error por rango de precio ===')
error_by_range

In [ ]:
# Error por marca (top 15 por frecuencia)
top_brands = test_df['brand'].value_counts().head(15).index
brand_errors = test_df[test_df['brand'].isin(top_brands)].groupby('brand').agg(
    count=('abs_error', 'count'),
    mae=('abs_error', 'mean'),
    mape=('pct_error', 'mean'),
).sort_values('mae', ascending=False).round(1)
brand_errors.columns = ['N', 'MAE ($)', 'MAPE (%)']

fig, ax = plt.subplots(figsize=(10, 6))
brand_errors['MAE ($)'].plot.barh(ax=ax, color='steelblue')
ax.set_title('MAE por marca (top 15)')
ax.set_xlabel('MAE ($)')
plt.tight_layout()
plt.savefig('../outputs/figures/error_by_brand.png', dpi=150)

## 3.6 Visualización del brand embedding

In [ ]:
from sklearn.decomposition import PCA

# Extraer embeddings aprendidos
embeddings = model.brand_embedding.weight.detach().cpu().numpy()
idx2brand = {v: k for k, v in brand2idx.items()}
brand_names = [idx2brand[i] for i in range(len(brand2idx))]

# Calcular precio medio por marca
brand_avg_price = df.groupby('brand')['price_clean'].mean()

# PCA a 2D
pca = PCA(n_components=2)
emb_2d = pca.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(12, 8))
prices = [brand_avg_price.get(b, 0) for b in brand_names]
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1], c=prices, cmap='RdYlGn_r', 
                     s=60, alpha=0.8, edgecolors='gray', linewidth=0.5)
plt.colorbar(scatter, label='Precio medio ($)')

# Anotar marcas representativas
for i, brand in enumerate(brand_names):
    if brand_avg_price.get(brand, 0) > 800 or brand_avg_price.get(brand, 0) < 80:
        ax.annotate(brand, (emb_2d[i, 0], emb_2d[i, 1]), fontsize=7, alpha=0.8)

ax.set_title('Brand Embeddings (PCA 2D) — color = precio medio')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} var)')
plt.tight_layout()
plt.savefig('../outputs/figures/brand_embedding_pca.png', dpi=150)

**Interpretación**: El modelo ha aprendido a organizar las marcas en el espacio de embeddings de forma coherente con su rango de precios. Las marcas de lujo (Montblanc, Versace) se agrupan lejos de las marcas económicas (Reflex Active, Fitbit). Esto confirma que el embedding está capturando señal real, no ruido.

## 3.7 Conclusiones

### Resultados principales

| Métrica | Valor (V4 final) |
|---------|------------------|
| R² (log) | **0.872** |
| R² (dollar) | **0.821** |
| MAE | **$76.57** |
| RMSE | **$174.33** |
| MAPE | **18.3%** |
| Parámetros | 226,201 / 250,000 (90%) |

### Factores clave de éxito

1. **Multi-input design**: Combinar CNN + brand embedding + text features fue determinante. El salto de V2 (solo imagen, R²=0.44) a V3 (+ marca, R²=0.75) demuestra que la marca es el predictor dominante.

2. **Iteración guiada por diagnóstico**: Cada versión respondió a un problema observado: underfitting (V2), falta de info contextual (V3), features textuales + preprocessing (V4).

3. **Depthwise separable convolutions**: Permitieron 4 bloques con hasta 256 filtros y dual conv, todo dentro de 250K params.

4. **CLAHE preprocessing**: Mejoró el contraste de detalles metálicos y textura de esfera, ayudando a la CNN a distinguir acabados.

### Patrones aprendidos (Grad-CAM)

- La CNN atiende a esfera, bisel, corona y brazalete — zonas con información de calidad de acabado.
- En chronographs, las subesferas reciben alta activación.
- El brand embedding separa marcas por rango de precio de forma coherente.

### Limitaciones

1. **Relojes de alta gama ($2,000+)**: Pocas muestras (~2% del dataset) dificultan predicciones precisas. El modelo subestima en este rango.
2. **Dependencia de la marca**: Sin la marca, el R² cae a ~0.44. La CNN sola no puede leer logos.
3. **250K params**: Limita la capacidad visual. Con transfer learning (no permitido aquí), los resultados serían superiores.
4. **Imágenes de producto**: Todas son fotos de catálogo con fondo blanco/gris. El modelo no generalizaría bien a fotos "in the wild".

### Posibles mejoras

1. **Oversampling de relojes caros**: Duplicar muestras $1K+ o aplicar sampling weights en el loss.
2. **Knowledge distillation**: Entrenar un modelo grande sin límite de params, luego destilar al modelo de 250K.
3. **Test-Time Augmentation (TTA)**: Promediar predicciones con múltiples augmentaciones del mismo input.
4. **Transfer learning**: Si el constraint lo permitiera, fine-tuning de EfficientNet-B0 (~5M params) daría R²>0.95.
5. **Datos adicionales**: Añadir el campo `name` completo como input textual (mini-BERT o TF-IDF).